# **1) SCREENSHOT COLLECTION**

## *1.1) Libraries*

In [ ]:
!pip install playwright
!playwright install chromium
!playwright install-deps
!pip install opencv-python Pillow numpy

## *1.2) Configuration & Helpers*

In [ ]:
import os
import asyncio
import nest_asyncio
import cv2
import numpy as np
import shutil
from datetime import datetime
from PIL import Image, ImageDraw, ImageFont
from playwright.async_api import async_playwright
from google.colab import drive, userdata

drive.mount('/content/drive')

# --- CONFIG ---
# Set BASE_URL in Colab Secrets (Tools > Secrets) as "BASE_URL"
BASE_URL       = userdata.get("BASE_URL")           # e.g. "https://your-app.example.com"
LOCAL_OUTPUT_DIR  = "/content/output"
DRIVE_OUTPUT_DIR  = "/content/drive/MyDrive/ExtractionProject/output"

os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

nest_asyncio.apply()


def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def slugify(text):
    """Sanitise a string so it is safe to use as a directory name."""
    if not text:
        return "unnamed"
    return "".join([c for c in text if c.isalnum() or c in (" ", ".", "_", "-")]).strip()


def get_unique_directory(local_base, drive_base):
    """
    Return versioned paths that do not yet exist on disk.
    Appends _v1, _v2 … until a free slot is found.
    This prevents overwriting already-captured screenshots.
    """
    counter = 1
    local_path, drive_path = local_base, drive_base
    while os.path.exists(local_path) or os.path.exists(drive_path):
        local_path = f"{local_base}_v{counter}"
        drive_path = f"{drive_base}_v{counter}"
        counter += 1
    return local_path, drive_path


def preprocess_and_stamp(file_path):
    """
    Apply CLAHE contrast enhancement + binarisation to improve LLM extraction accuracy,
    then embed a capture timestamp watermark into the bottom-right corner.
    """
    img_cv = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
    if img_cv is not None:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(img_cv)
        _, thresh = cv2.threshold(enhanced, 210, 255, cv2.THRESH_BINARY)
        cv2.imwrite(file_path, thresh)

    try:
        img = Image.open(file_path)
        draw = ImageDraw.Draw(img)
        timestamp = datetime.now().strftime("%d.%m.%Y %H:%M")
        label = f"Captured: {timestamp}"
        try:
            font = ImageFont.truetype(
                "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 15
            )
        except Exception:
            font = ImageFont.load_default()

        bbox = draw.textbbox((0, 0), label, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        margin = 30
        x = img.size[0] - tw - margin
        y = img.size[1] - th - margin
        draw.rectangle([x - 10, y - 10, x + tw + 10, y + th + 10], fill=255)
        draw.text((x, y), label, font=font, fill=0)
        img.save(file_path)
    except Exception as e:
        log(f"Image processing error: {e}")


## *1.3) Web Scraping & Screenshot Runner*

In [ ]:
async def run():
    """
    Authenticate, iterate over target entities, navigate each form page,
    capture a full-height screenshot, pre-process the image, and sync to Drive.

    Authentication credentials and target identifiers are read exclusively
    from Colab Secrets — no credentials appear in this notebook.

    Idempotency: if a screenshot already exists for a form, the step is skipped
    (see `get_unique_directory` and the `check_file_drive` guard below).
    """
    app_username = userdata.get("APP_USERNAME")
    app_password = userdata.get("APP_PASSWORD")
    # Comma-separated list of entity IDs to process, e.g. "E001,E002,E003"
    target_ids   = [x.strip() for x in userdata.get("TARGET_IDS").split(",")]

    async with async_playwright() as p:
        log("Launching browser …")
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(viewport={"width": 1920, "height": 1080})
        page    = await context.new_page()

        try:
            # ── Authentication ──────────────────────────────────────────────
            await page.goto(f"{BASE_URL}/login", wait_until="networkidle")
            await page.fill("input[name='username']", app_username)
            await page.fill("input[name='password']", app_password)
            await page.click("button[type='submit']")
            await page.wait_for_load_state("networkidle")

            # ── Entity iteration ────────────────────────────────────────────
            for entity_id in target_ids:
                log(f"--- Processing entity: {entity_id} ---")
                current_page_num = 1
                found = False

                while not found:
                    await page.goto(
                        f"{BASE_URL}/entities?page={current_page_num}",
                        wait_until="networkidle",
                    )
                    rows = page.locator(".entity-list .row")

                    if await rows.count() == 0:
                        break

                    for row in await rows.all():
                        code_el = (await row.locator(".entity-code").first.inner_text()).strip()
                        if code_el == entity_id:
                            found = True
                            log(f"  Entity found on page {current_page_num}: {entity_id}")

                            await row.locator(".entity-link a").first.click()
                            await page.wait_for_load_state("networkidle")

                            # ── Visit / section iteration ───────────────────
                            visit_items = await page.locator("ul.visits-nav > li").all()

                            for visit in visit_items:
                                v_status = await visit.get_attribute("title")
                                if v_status == "Not started":
                                    continue

                                v_link = visit.locator("a").first
                                v_text = (await v_link.inner_text()).split("\n")[0].strip()
                                log(f"    Visit: {v_text}")
                                await v_link.click()
                                await asyncio.sleep(1)

                                sub_items = await visit.locator("ul.formsMenu > li").all()
                                current_group = ""

                                for li in sub_items:
                                    header = li.locator("strong").first
                                    if await header.count() > 0:
                                        current_group = slugify(await header.inner_text())
                                        continue

                                    form_link   = li.locator("a").first
                                    if await form_link.count() == 0:
                                        continue

                                    form_text   = (await form_link.inner_text()).strip()
                                    form_slug   = slugify(form_text)
                                    form_status = await form_link.get_attribute("title")

                                    # Only capture forms that have data
                                    if form_status not in ("Verified", "Completed"):
                                        log(f"      [SKIP] ({form_status}): {form_text}")
                                        continue

                                    path_parts = [slugify(entity_id), slugify(v_text)]
                                    if current_group:
                                        path_parts.append(current_group)
                                    path_parts.append(form_slug)

                                    local_base = os.path.join(LOCAL_OUTPUT_DIR, *path_parts)
                                    drive_base = os.path.join(DRIVE_OUTPUT_DIR, *path_parts)
                                    check_drive = os.path.join(
                                        drive_base, f"{os.path.basename(drive_base)}.png"
                                    )

                                    # ── Idempotency guard ───────────────────
                                    if os.path.exists(check_drive):
                                        log(f"      [EXISTS] Skipping: {form_text}")
                                        continue

                                    final_local, _ = get_unique_directory(local_base, drive_base)
                                    os.makedirs(final_local, exist_ok=True)
                                    target_path = os.path.join(
                                        final_local, f"{os.path.basename(final_local)}.png"
                                    )

                                    log(f"      [CAPTURE] ({form_status}): {form_text}")
                                    await form_link.click()
                                    await page.wait_for_load_state("networkidle")

                                    try:
                                        container = page.locator("div.form-container").first
                                        await container.wait_for(state="visible", timeout=7000)
                                        await container.screenshot(path=target_path)
                                        preprocess_and_stamp(target_path)
                                    except Exception:
                                        log(f"      [ERROR] Screenshot failed: {form_text}")

                            log(f"  Syncing to Drive …")
                            os.system(f"cp -rv {LOCAL_OUTPUT_DIR}/* {DRIVE_OUTPUT_DIR}/")
                            shutil.rmtree(LOCAL_OUTPUT_DIR)
                            os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
                            log(f"  Done: {entity_id}")
                            break

                    if not found:
                        next_link = page.locator(
                            f"ul.pagination li a[href*='page={current_page_num + 1}']"
                        ).first
                        if await next_link.count() > 0:
                            current_page_num += 1
                        else:
                            log(f"  [NOT FOUND] {entity_id}")
                            break

        except Exception as e:
            log(f"Pipeline error: {e}")
        finally:
            log("Closing browser.")
            await browser.close()


await run()


# **2) FILE DISCOVERY & SNAPSHOT**

## *2.1) Libraries*

In [ ]:
import os
import json
from google.colab import drive

## *2.2) Discovery Logic*

Walks the output directory and builds a `pending_tasks.json` job queue.
Each entry holds the relative path of a PNG that has no corresponding `.xlsx` yet.
If the snapshot file already exists the step is skipped, ensuring **idempotency**.

In [ ]:
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT         = "/content/drive/MyDrive/ExtractionProject/output"
PENDING_TASKS_FILE = "pending_tasks.json"

if not os.path.exists(PENDING_TASKS_FILE):
    tasks = []
    for root, _, files in os.walk(DRIVE_ROOT):
        for f in files:
            if f.lower().endswith(".png"):
                base      = os.path.splitext(f)[0]
                png_path  = os.path.join(root, f)
                xlsx_path = os.path.join(root, f"{base}.xlsx")
                if not os.path.exists(xlsx_path):
                    rel_path = os.path.relpath(png_path, DRIVE_ROOT)
                    tasks.append({"rel_path": rel_path, "file_uri": None})

    with open(PENDING_TASKS_FILE, "w") as fh:
        json.dump(tasks, fh, indent=2)
    print(f"[DISCOVERY] Snapshot created — {len(tasks)} pending tasks.")
else:
    print("[DISCOVERY] Snapshot already exists. Skipping regeneration.")


# **3) FILES API UPLOAD**

## *3.1) Libraries*

In [ ]:
import json
import os
from google.genai import Client
from google.colab import userdata, drive

## *3.2) Idempotent Upload*

Uploads each PNG in the job queue to the Gemini Files API.
If a `file_uri` is already recorded for a task the upload is skipped,
making repeated runs safe and cost-free for already-uploaded images.

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT         = "/content/drive/MyDrive/ExtractionProject/output"
PENDING_TASKS_FILE = "pending_tasks.json"

client = Client(api_key=userdata.get('GEMINI_API_KEY'))


def _print_progress(done, total, success, skipped, failed):
    pct    = (done / total) * 100
    bar_n  = int(30 * done / total)
    bar    = "█" * bar_n + "-" * (30 - bar_n)
    print(
        f"\r[{bar}] {pct:6.2f}%"
        f" | Done: {done}/{total}"
        f" | OK: {success}"
        f" | Skip: {skipped}"
        f" | Fail: {failed}",
        end="",
    )


def run_upload_process():
    if not os.path.exists(PENDING_TASKS_FILE):
        print(f"[ERROR] {PENDING_TASKS_FILE} not found. Run Section 2 first.")
        return

    with open(PENDING_TASKS_FILE, "r", encoding="utf-8") as fh:
        tasks = json.load(fh)

    total     = len(tasks)
    processed = success = failed = skipped = 0
    updated   = False

    print("=" * 50)
    print(" " * 17 + "UPLOAD PROGRESS")
    print("=" * 50)

    for task in tasks:
        processed += 1

        if task.get("file_uri"):          # already uploaded → skip
            skipped += 1
            _print_progress(processed, total, success, skipped, failed)
            continue

        abs_path = os.path.join(DRIVE_ROOT, task["rel_path"])
        if not os.path.exists(abs_path):
            failed += 1
            _print_progress(processed, total, success, skipped, failed)
            continue

        try:
            with open(abs_path, "rb") as img_file:
                up = client.files.upload(file=img_file, config={"mime_type": "image/png"})
            task["file_uri"] = up.uri
            success += 1
            updated  = True
        except Exception:
            failed += 1

        _print_progress(processed, total, success, skipped, failed)

    print("\n")

    if updated:
        with open(PENDING_TASKS_FILE, "w", encoding="utf-8") as fh:
            json.dump(tasks, fh, indent=2)
        print("[INFO] pending_tasks.json updated.")
    else:
        print("[INFO] No new files to upload.")

    print("\nUPLOAD SUMMARY")
    print("-" * 40)
    print(f"SUCCESS : {success}")
    print(f"SKIPPED : {skipped}")
    print(f"FAILED  : {failed}")
    print("-" * 40)


run_upload_process()


# **4) BATCH PREPARATION & SUBMISSION**

## *4.1) Libraries*

In [ ]:
import json, os
from google import genai
from google.genai import types
from google.colab import userdata

## *4.2) Safe Submission*

Builds the JSONL batch input file from the job queue, then submits it to the
Gemini Batch API. An existing active job is detected before a new one is created.

In [ ]:
client = genai.Client(
    api_key=userdata.get('GEMINI_API_KEY'),
    http_options=types.HttpOptions(api_version='v1beta'),
)

BATCH_INPUT_FILE = "batch_input.jsonl"
JOB_ID_FILE      = "job_id.txt"
MODEL_NAME = "gemini-3-flash-preview"

PROMPT = """
You are a document form extraction AI.

Your task is to extract ONLY the values that are visibly filled in the form image.
The form is a structured data-entry form with sections, checkboxes, radio buttons,
numeric inputs, dates, calculated fields, and repeated sub-forms.

GENERAL RULES:
- DO NOT guess or infer missing values.
- If a field is empty or not selected, return null.
- Extract values exactly as shown (numbers, decimals, dates, units).
- Ignore instructions, help text, headers, footers, and metadata unless they contain filled values.
- Ignore capture timestamps, page numbers, and audit text.

OUTPUT FORMAT:
- Return a JSON ARRAY only.
- Each item must be a flat object:
  {
    "section": "<section title>",
    "field":   "<exact field label>",
    "value":   "<extracted value or null>"
  }
- Do NOT return nested objects.
- Do NOT include any text outside the JSON array.

FIELD-SPECIFIC RULES:

1) CHECKBOX / RADIO BUTTONS
   - Return the selected option label exactly as shown (e.g., "Yes", "No").
   - Multiple selections → return an array of selected labels.
   - Nothing selected → null.

2) DATE FIELDS
   - Combine day, month, year → DD.MM.YYYY.
   - Any part missing → null.

3) TIME FIELDS
   - Combine hour and minute → HH:MM (24-hour).
   - Any part missing → null.

4) NUMERIC FIELDS
   - Extract exactly as shown (including decimals).
   - Do NOT calculate or normalise.
   - Do NOT include units in the value unless they are part of the input box.

5) REPEATED / INDEXED FIELDS (e.g., Item 1, Item 2)
   - Treat each repeated item as a separate field.
   - Include the index in the field name: "Item 1 Description", "Item 2 Description".

6) CALCULATED / READ-ONLY FIELDS
   - Extract visible calculated values as-is. Do NOT recompute.

7) SECTION HANDLING
   - Use the visible section title as the "section" value.
   - No clear title → use "General".

IMPORTANT:
- Preserve field order top-to-bottom.
- Return STRICT JSON. No markdown, no explanations.

REMINDER: Extract ONLY what is visible and filled. Never guess.
"""

# ── Build JSONL input ─────────────────────────────────────────────────────────
if not os.path.exists(BATCH_INPUT_FILE):
    print(f"[INFO] Building {BATCH_INPUT_FILE} …")
    with open("pending_tasks.json", "r") as fh:
        tasks = json.load(fh)

    with open(BATCH_INPUT_FILE, "w") as jf:
        for task in tasks:
            if task.get("file_uri"):
                req = {
                    "custom_id": task["rel_path"],
                    "request": {
                        "contents": [
                            {
                                "role": "user",
                                "parts": [
                                    {"text": PROMPT},
                                    {"file_data": {
                                        "mime_type": "image/png",
                                        "file_uri": task["file_uri"],
                                    }},
                                ],
                            }
                        ],
                        "generation_config": {
                            "temperature": 0,
                            "response_mime_type": "application/json",
                        },
                    },
                }
                jf.write(json.dumps(req) + "\n")
    print(f"[OK] {BATCH_INPUT_FILE} ready.")

# ── Check for existing job ────────────────────────────────────────────────────
existing_job = None
if os.path.exists(JOB_ID_FILE):
    with open(JOB_ID_FILE) as fh:
        job_id = fh.read().strip()
    try:
        existing_job = client.batches.get(name=job_id)
        print(f"[BATCH] Existing job: {existing_job.name}  state: {existing_job.state}")
    except Exception as e:
        print(f"[BATCH] Could not retrieve saved job (may have expired): {e}")

# ── Submit new job if needed ──────────────────────────────────────────────────
if not existing_job or str(existing_job.state) in ("SUCCEEDED", "FAILED", "CANCELLED"):
    if existing_job and "SUCCEEDED" in str(existing_job.state):
        print("[BATCH] Previous job succeeded. Delete job_id.txt to start a fresh run.")
    else:
        print("[BATCH] Submitting new batch job …")

        uploaded_input = client.files.upload(
            file=BATCH_INPUT_FILE,
            config={"mime_type": "application/x-jsonl"},
        )
        print(f"[FILE] Input file uploaded: {uploaded_input.name}")

        new_job = client.batches.create(model=MODEL_NAME, src=uploaded_input.name)

        with open(JOB_ID_FILE, "w") as fh:
            fh.write(new_job.name)

        print(f"[OK] Job created: {new_job.name}")
        print("[INFO] Re-run this cell (or Section 4.3) to check status.")


## *4.3) Batch Status Monitor*

In [ ]:
from datetime import timedelta

try:
    client
except NameError:
    client = genai.Client(
        api_key=userdata.get('GEMINI_API_KEY'),
        http_options=types.HttpOptions(api_version='v1beta'),
    )

for job in client.batches.list():
    state_str = str(job.state).replace("JobState.JOB_STATE_", "").replace("JobState.", "")

    queued_at = (job.create_time + timedelta(hours=3)).strftime('%Y-%m-%d %H:%M:%S')

    raw_start = getattr(job, 'start_time', None)
    started_at = (
        (raw_start + timedelta(hours=3)).strftime('%Y-%m-%d %H:%M:%S')
        if raw_start else ("Completed" if state_str == "SUCCEEDED" else "Queued")
    )

    end_raw  = getattr(job, 'update_time', None)
    ended_at = (end_raw + timedelta(hours=3)).strftime('%Y-%m-%d %H:%M:%S') if end_raw else "—"

    print(f"Job   : {job.name}")
    print(f"State : {state_str}")
    if state_str == "PENDING":
        print(f"  Queued at : {queued_at}  (waiting in queue …)")
    elif state_str == "RUNNING":
        print(f"  Queued at : {queued_at}  |  Started : {started_at}  (running …)")
    else:
        print(f"  Queued at : {queued_at}  |  Started : {started_at}  |  Ended : {ended_at}")
    print("-" * 80)


# **5) OUTPUT DOWNLOAD & EXCEL GENERATION**

## *5.1) Libraries*

In [ ]:
import json, os
import pandas as pd
from google.genai import Client
from google.colab import userdata, drive

## *5.2) Robust Processing*

Downloads the completed batch results, parses each JSON response,
and writes an individual `.xlsx` file next to the source screenshot.
Already-converted files are skipped (idempotent).

In [ ]:
# Paste the active job name printed by Section 4.3
# e.g. "batches/xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
BATCH_ID = "batches/YOUR_BATCH_ID_HERE"

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

BATCH_OUTPUT_FILE = "batch_results.jsonl"
DRIVE_ROOT        = "/content/drive/MyDrive/ExtractionProject/output"

client = Client(api_key=userdata.get('GEMINI_API_KEY'))


def process_pipeline():
    # ── 1. Download results ──────────────────────────────────────────────────
    if not os.path.exists(BATCH_OUTPUT_FILE):
        if not BATCH_ID or "YOUR_BATCH_ID_HERE" in BATCH_ID:
            print("[ERROR] Set BATCH_ID before running.")
            return

        try:
            job       = client.batches.get(name=BATCH_ID)
            state_str = str(job.state).split('.')[-1]
            print(f"[STATUS] {state_str}")

            if any(s in state_str for s in ("RUNNING", "PENDING")):
                print("[INFO] Batch still processing — try again later.")
                return
            if "FAILED" in state_str:
                print("[ERROR] Batch FAILED.")
                return
            if "SUCCEEDED" in state_str:
                if not getattr(job, "dest", None) or not job.dest.file_name:
                    print("[ERROR] Batch succeeded but output file info is missing.")
                    return
                print(f"[DOWNLOAD] {job.dest.file_name}")
                content = client.files.download(file=job.dest.file_name)
                with open(BATCH_OUTPUT_FILE, "wb") as fh:
                    fh.write(content)
                print("[OK] batch_results.jsonl saved.")
        except Exception as e:
            print(f"[ERROR] {e}")
            return
    else:
        print("[INFO] Existing batch_results.jsonl found — skipping download.")

    # ── 2. Generate Excel files ──────────────────────────────────────────────
    processed = skipped = 0

    with open(BATCH_OUTPUT_FILE, "r", encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            try:
                res = json.loads(line)
            except Exception:
                continue

            rel_path = res.get("custom_id")
            if not rel_path:
                continue

            target_xlsx = os.path.join(DRIVE_ROOT, rel_path.replace(".png", ".xlsx"))

            if os.path.exists(target_xlsx):
                skipped += 1
                continue

            try:
                parts    = res.get("response", {}).get("candidates", [{}])[0]
                raw_text = "".join(
                    p.get("text", "") for p in parts.get("content", {}).get("parts", [])
                )
                if not raw_text:
                    print(f"[EMPTY] No content: {rel_path}")
                    continue

                clean = raw_text.strip()
                if clean.startswith("```"):
                    lines = clean.splitlines()
                    clean = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:]).strip()
                if clean.lower().startswith("json"):
                    clean = clean[4:].strip()

                data = json.loads(clean)
                df   = pd.DataFrame([data] if isinstance(data, dict) else data)

                os.makedirs(os.path.dirname(target_xlsx), exist_ok=True)
                df.to_excel(target_xlsx, index=False)
                print(f"[OK] {rel_path}")
                processed += 1
            except Exception as e:
                print(f"[FAIL] {rel_path} — {e}")

    print(f"\n[DONE] {processed} generated, {skipped} skipped.")


process_pipeline()


# **6) TOKEN USAGE & COST ANALYSIS**

In [ ]:
import json, os, tempfile
from google.genai import Client
from google.colab import userdata

BATCH_ID = "batches/YOUR_BATCH_ID_HERE"

client = Client(api_key=userdata.get('GEMINI_API_KEY'))

PRICE_INPUT_1M  = 0.25   # USD per 1M input tokens
PRICE_OUTPUT_1M = 1.50   # USD per 1M output tokens


def analyze_tokens(file_path):
    total_prompt = total_output = total_thoughts = total_tokens = image_count = 0

    with open(file_path, "r", encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            try:
                usage = json.loads(line).get("response", {}).get("usageMetadata", {})
                total_prompt  += usage.get("promptTokenCount", 0)
                total_output  += usage.get("candidatesTokenCount", 0)
                total_thoughts+= usage.get("thoughtsTokenCount", 0)
                total_tokens  += usage.get("totalTokenCount", 0)
                image_count   += 1
            except Exception:
                continue

    if image_count == 0:
        print("[INFO] No usage data found.")
        return

    combined_out = total_output + total_thoughts
    cost_in      = (total_prompt  / 1_000_000) * PRICE_INPUT_1M
    cost_out     = (combined_out  / 1_000_000) * PRICE_OUTPUT_1M
    total_cost   = cost_in + cost_out

    print("\n" + "=" * 48)
    print("       TOKEN USAGE & COST REPORT")
    print("=" * 48)
    print(f"  Images processed       : {image_count}")
    print("-" * 48)
    print(f"  Input tokens           : {total_prompt:,}")
    print(f"  Output tokens (incl.)  : {combined_out:,}")
    print(f"  Total tokens           : {total_tokens:,}")
    print("-" * 48)
    print(f"  Input cost             : ${cost_in:.6f}")
    print(f"  Output cost            : ${cost_out:.6f}")
    print(f"  TOTAL COST             : ${total_cost:.6f}")
    print("-" * 48)
    print(f"  Avg tokens / image     : {total_tokens / image_count:.1f}")
    print(f"  Avg cost   / image     : ${total_cost / image_count:.6f}")
    print("=" * 48 + "\n")


def run_cost_analysis(batch_id):
    if not batch_id or "YOUR_BATCH_ID_HERE" in batch_id:
        print("[ERROR] Set BATCH_ID before running.")
        return
    try:
        job = client.batches.get(name=batch_id)
        if "SUCCEEDED" not in str(job.state):
            print(f"[INFO] Batch not yet completed: {job.state}")
            return
        if not getattr(job, "dest", None) or not job.dest.file_name:
            return

        with tempfile.NamedTemporaryFile(mode="wb+", delete=False, suffix=".jsonl") as tmp:
            tmp.write(client.files.download(file=job.dest.file_name))
            tmp_path = tmp.name

        analyze_tokens(tmp_path)
        os.remove(tmp_path)
    except Exception as e:
        print(f"[ERROR] {e}")


run_cost_analysis(BATCH_ID)


# **7) LINE LISTING — HISTORY**

Aggregates all *History* form Excel files across entities and visits
into a single consolidated listing sheet.
Adapt the column mapping below to match your form's field labels.

## *7.1) Libraries*

In [ ]:
!pip install -q openpyxl

## *7.2) Data Processing*

In [ ]:
import os, glob, re, json
import pandas as pd
import openpyxl
from google.colab import drive
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

drive.mount('/content/drive', force_remount=True)

BASE_DIR     = '/content/drive/MyDrive/ExtractionProject/output/'
OUTPUT_DIR   = '/content/drive/MyDrive/ExtractionProject/Line_Listings/'
OUTPUT_FILE  = os.path.join(OUTPUT_DIR, 'Master_Line_Listing.xlsx')
SHEET_NAME   = 'History_Listing'

os.makedirs(OUTPUT_DIR, exist_ok=True)

COLUMNS = ['Subject', 'Visit', 'FormDate', 'Entry#',
           'History Term', 'Start Date', 'End Date', 'Ongoing']

# ── Field label mapping ──────────────────────────────────────────────────────
# Map your form's exact field labels to the canonical column names above.
# Update these strings to match the fields in your own form.
FIELD_MAP = {
    "diagnosis":  "History Term",
    "onset date": "Start Date",
    "ongoing":    "Ongoing",
    "end date":   "End Date",
}


def get_entry_num(field_text):
    match = re.search(r'[:\*]?\s*(\d+)$', str(field_text).strip())
    return int(match.group(1)) if match else 1


def parse_history_file(filepath, subject_id, form_date):
    try:
        df = pd.read_excel(filepath)
        if df.empty or 'field' not in df.columns or 'value' not in df.columns:
            return []
    except Exception:
        return []

    records = {}

    for _, row in df.iterrows():
        field = str(row['field']).strip()
        value = "" if pd.isna(row['value']) else row['value']
        entry = get_entry_num(field)
        field_lower = field.lower()

        matched_col = next(
            (col for kw, col in FIELD_MAP.items() if kw in field_lower), None
        )
        if not matched_col:
            continue

        if entry not in records:
            records[entry] = {
                'Subject': subject_id, 'Visit': 'Screening',
                'FormDate': form_date, 'Entry#': entry,
                'History Term': '', 'Start Date': '',
                'End Date': '', 'Ongoing': '',
            }

        records[entry][matched_col] = value

    final = []
    for num in sorted(records):
        rec = records[num]
        if not rec['History Term']:
            continue
        if str(rec['Ongoing']).strip().lower() == 'yes':
            rec['End Date'] = ''
        final.append(rec)
    return final


def main():
    all_records = []

    for subject_id in os.listdir(BASE_DIR):
        subj_path = os.path.join(BASE_DIR, subject_id)
        if not os.path.isdir(subj_path):
            continue

        # Walk all visit folders and look for any folder named "*history*"
        for visit_folder in os.listdir(subj_path):
            visit_path = os.path.join(subj_path, visit_folder)
            for form_folder in os.listdir(visit_path):
                if 'history' not in form_folder.lower():
                    continue
                form_path  = os.path.join(visit_path, form_folder)
                date_dirs  = [d for d in os.listdir(form_path)
                              if os.path.isdir(os.path.join(form_path, d))]
                for date_d in date_dirs:
                    for xf in glob.glob(os.path.join(form_path, date_d, '*.xlsx')):
                        all_records.extend(parse_history_file(xf, subject_id, date_d))

    if not all_records:
        print("[INFO] No records found.")
        return

    df_out = pd.DataFrame(all_records, columns=COLUMNS)
    mode   = 'a' if os.path.exists(OUTPUT_FILE) else 'w'
    kw     = {'if_sheet_exists': 'replace'} if mode == 'a' else {}

    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl', mode=mode, **kw) as writer:
        df_out.to_excel(writer, sheet_name=SHEET_NAME, index=False)

    wb = load_workbook(OUTPUT_FILE)
    ws = wb[SHEET_NAME]
    ws.auto_filter.ref = f"A1:{openpyxl.utils.get_column_letter(ws.max_column)}{ws.max_row}"
    wb.save(OUTPUT_FILE)

    print(f"[OK] {len(all_records)} rows written to '{SHEET_NAME}' in {OUTPUT_FILE}")


main()


# **8) LINE LISTING — MEDICATION**

Aggregates all *Medication* form Excel files into a consolidated listing sheet.
Versioned date folders (e.g. `08.07.2025_v1`) are normalised before grouping.

## *8.1) Libraries*

In [ ]:
!pip install -q openpyxl

## *8.2) Data Processing*

In [ ]:
import os, glob, re
import pandas as pd
import openpyxl
from google.colab import drive
from openpyxl import load_workbook

drive.mount('/content/drive', force_remount=True)

BASE_DIR    = '/content/drive/MyDrive/ExtractionProject/output/'
OUTPUT_DIR  = '/content/drive/MyDrive/ExtractionProject/Line_Listings/'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'Master_Line_Listing.xlsx')
SHEET_NAME  = 'Medication_Listing'

os.makedirs(OUTPUT_DIR, exist_ok=True)

COLUMNS = [
    'Subject', 'Visit', 'FormDate', 'Med#',
    'Trade Name', 'INN', 'Indication', 'Dose', 'Unit',
    'Frequency', 'Route', 'Dose Form', 'Start Date', 'End Date', 'Ongoing',
]

# ── Field label mapping ──────────────────────────────────────────────────────
# Update keywords to match the exact field labels in your own form.
MED_FIELD_MAP = {
    "trade name":            "Trade Name",
    "inn":                   "INN",
    "indication":            "Indication",
    "individual dose":       "Dose",
    "units":                 "Unit",
    "frequency":             "Frequency",
    "route of administration": "Route",
    "dose form":             "Dose Form",
    "start date":            "Start Date",
    "end date":              "End Date",
    "ongoing":               "Ongoing",
}


def clean_date_folder(name):
    """Strip version suffixes such as _v1, _v2 from date folder names."""
    return re.split(r'_v\d+', name)[0]


def process_medication_files():
    all_meds     = []
    counters     = {}  # {(subject, visit, clean_date): last_med_num}

    for subject in sorted(os.listdir(BASE_DIR)):
        subj_path = os.path.join(BASE_DIR, subject)
        if not os.path.isdir(subj_path):
            continue

        for visit in os.listdir(subj_path):
            visit_path = os.path.join(subj_path, visit)

            for form_folder in os.listdir(visit_path):
                if 'medication' not in form_folder.lower():
                    continue
                med_path    = os.path.join(visit_path, form_folder)
                date_folders= sorted(
                    d for d in os.listdir(med_path)
                    if os.path.isdir(os.path.join(med_path, d))
                )

                for date_f in date_folders:
                    form_date  = clean_date_folder(date_f)
                    key        = (subject, visit, form_date)
                    counters.setdefault(key, 0)

                    for xf in glob.glob(os.path.join(med_path, date_f, '*.xlsx')):
                        try:
                            df = pd.read_excel(xf)
                            if df.empty:
                                continue
                        except Exception:
                            continue

                        current_med = None

                        for _, row in df.iterrows():
                            field   = str(row['field']).strip()
                            value   = str(row['value']).strip() if pd.notna(row['value']) else ''
                            f_lower = field.lower()

                            # Skip consent / participation-level questions
                            if 'has the participant taken' in f_lower:
                                continue

                            # New medication block starts with "Trade Name" field
                            if re.search(r'\btrade\s+name\b', f_lower) and 'inn' not in f_lower:
                                if current_med and (current_med['Trade Name'] or current_med['INN']):
                                    all_meds.append(current_med)
                                counters[key] += 1
                                current_med = {col: '' for col in COLUMNS}
                                current_med.update({
                                    'Subject': subject, 'Visit': visit,
                                    'FormDate': form_date,
                                    'Med#': counters[key],
                                    'Trade Name': value,
                                })
                                continue

                            if current_med:
                                matched = next(
                                    (col for kw, col in MED_FIELD_MAP.items() if kw in f_lower),
                                    None,
                                )
                                if matched:
                                    if matched == 'Ongoing' and value.lower() == 'yes':
                                        current_med['End Date'] = ''
                                    elif matched == 'End Date' and current_med.get('Ongoing', '').lower() == 'yes':
                                        pass  # don't overwrite cleared end date
                                    else:
                                        current_med[matched] = value

                        if current_med:
                            all_meds.append(current_med)

    if not all_meds:
        print("[INFO] No medication records found.")
        return

    df_out = pd.DataFrame(all_meds)
    mode   = 'a' if os.path.exists(OUTPUT_FILE) else 'w'
    kw     = {'if_sheet_exists': 'replace'} if mode == 'a' else {}

    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl', mode=mode, **kw) as writer:
        df_out.to_excel(writer, sheet_name=SHEET_NAME, index=False)

    wb = load_workbook(OUTPUT_FILE)
    ws = wb[SHEET_NAME]
    ws.auto_filter.ref = ws.dimensions
    wb.save(OUTPUT_FILE)

    print(f"[OK] {len(all_meds)} medication records written to '{SHEET_NAME}'")


process_medication_files()


# **9) LINE LISTING — EVENTS**

Aggregates all *Event* form Excel files into a consolidated listing sheet.
An event form is identified by a configurable discriminator field (see `EVENT_DISCRIMINATOR`).

## *9.1) Libraries*

In [ ]:
!pip install -q openpyxl

## *9.2) Data Processing*

In [ ]:
import os, re
import pandas as pd
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

BASE_PATH        = '/content/drive/MyDrive/ExtractionProject/output/'
OUTPUT_DIR       = '/content/drive/MyDrive/ExtractionProject/Line_Listings/'
MASTER_FILE_PATH = os.path.join(OUTPUT_DIR, 'Master_Line_Listing.xlsx')
SHEET_NAME       = 'Events_Listing'

os.makedirs(OUTPUT_DIR, exist_ok=True)

COLUMNS = [
    'Subject', 'Visit', 'Event Term', 'Severity',
    'Start Date', 'End Date', 'Outcome', 'Action Taken', 'Notes',
]

# ── Discriminator ─────────────────────────────────────────────────────────────
# A form is treated as an Event form only if this field/value pair is present.
# Update to match the actual discriminator in your form.
EVENT_DISCRIMINATOR_FIELD = "type of event"
EVENT_DISCRIMINATOR_VALUE = "adverse event"

# ── Field label mapping ───────────────────────────────────────────────────────
# Update keywords to match the exact field labels in your own form.
EVENT_FIELD_MAP = {
    "describe the event":    "Event Term",
    "specify the event":     "Event Term",
    "severity":              "Severity",
    "end date":              "End Date",
    "outcome":               "Outcome",
    "action taken":          "Action Taken",
    "medication":            "Notes",
    "prescribed drug":       "Notes",
}


def normalise(text):
    if pd.isna(text):
        return ""
    return re.sub(r'[^a-z0-9\s]', '', str(text).lower()).strip()


def process_event_files():
    all_events = []

    for subject_id in os.listdir(BASE_PATH):
        subj_path = os.path.join(BASE_PATH, subject_id)
        if not os.path.isdir(subj_path):
            continue

        for visit in os.listdir(subj_path):
            for event_folder in os.listdir(os.path.join(subj_path, visit)):
                event_path = os.path.join(subj_path, visit, event_folder)
                if not os.path.isdir(event_path):
                    continue

                for date_folder in os.listdir(event_path):
                    folder_path = os.path.join(event_path, date_folder)
                    for fname in os.listdir(folder_path):
                        if not fname.endswith('.xlsx'):
                            continue

                        try:
                            df = pd.read_excel(os.path.join(folder_path, fname))
                        except Exception:
                            continue

                        # Discriminator check
                        is_event_form = any(
                            EVENT_DISCRIMINATOR_FIELD in normalise(r['field'])
                            and EVENT_DISCRIMINATOR_VALUE in str(r['value']).lower()
                            for _, r in df.iterrows()
                        )
                        if not is_event_form:
                            continue

                        tmp_date = tmp_time = ''
                        current_event = None

                        for _, row in df.iterrows():
                            field = normalise(row['field'])
                            value = str(row['value']).strip() if pd.notna(row['value']) else ''

                            if field == 'date':
                                tmp_date = value
                            elif field == 'time':
                                tmp_time = value

                            matched_col = next(
                                (col for kw, col in EVENT_FIELD_MAP.items() if kw in field), None
                            )

                            if matched_col == 'Event Term':
                                if current_event:
                                    all_events.append(current_event)
                                current_event = {col: '' for col in COLUMNS}
                                current_event.update({
                                    'Subject':    subject_id,
                                    'Visit':      visit,
                                    'Event Term': value,
                                    'Start Date': f"{tmp_date} {tmp_time}".strip(),
                                })
                            elif current_event and matched_col:
                                if matched_col == 'End Date':
                                    if 'ongoing' in field and 'yes' in value.lower():
                                        current_event['End Date'] = 'Ongoing'
                                    elif current_event['End Date'] != 'Ongoing':
                                        current_event['End Date'] = value
                                elif matched_col == 'Notes':
                                    existing = current_event['Notes']
                                    current_event['Notes'] = f"{existing} | {value}".strip(' |') if existing else value
                                else:
                                    current_event[matched_col] = value

                        if current_event:
                            all_events.append(current_event)

    if not all_events:
        print("[INFO] No event records found.")
        return

    df_out = pd.DataFrame(all_events)
    mode   = 'a' if os.path.exists(MASTER_FILE_PATH) else 'w'
    kw     = {'if_sheet_exists': 'replace'} if mode == 'a' else {}

    with pd.ExcelWriter(MASTER_FILE_PATH, engine='openpyxl', mode=mode, **kw) as writer:
        df_out.to_excel(writer, sheet_name=SHEET_NAME, index=False)
        writer.sheets[SHEET_NAME].auto_filter.ref = writer.sheets[SHEET_NAME].dimensions

    print(f"[OK] {len(all_events)} event records written to '{SHEET_NAME}'")


process_event_files()
